# OptiCore Quickstart

This notebook covers the core features of OptiCore in 5 minutes:
1. Price a European option
2. Solve for implied volatility
3. Compute all Greeks
4. Vectorized batch pricing
5. Visualize Greeks profiles and payoff diagrams

In [ ]:
import opticore as oc
import numpy as np

print(f"OptiCore version: {oc.__version__}")

## 1. Price a European Call

Using Black-Scholes-Merton with:
- Spot = $100, Strike = $105
- 6 months to expiry, 5% risk-free rate
- 20% annualized volatility

In [ ]:
call_price = oc.price(spot=100, strike=105, expiry=0.5, rate=0.05, vol=0.20, kind="call")
put_price  = oc.price(spot=100, strike=105, expiry=0.5, rate=0.05, vol=0.20, kind="put")

print(f"Call price: ${call_price:.4f}")
print(f"Put  price: ${put_price:.4f}")

# Verify put-call parity: C - P = S - K*exp(-rT)
parity_lhs = call_price - put_price
parity_rhs = 100 - 105 * np.exp(-0.05 * 0.5)
print(f"\nPut-call parity check: {parity_lhs:.6f} = {parity_rhs:.6f} ✓")

## 2. Implied Volatility

Given a market price, solve for the volatility that produces it.
Uses Jaeckel's "Let's Be Rational" — full 64-bit precision in 2 iterations.

In [ ]:
# Start with a known vol, price it, then solve back
true_vol = 0.25
market_price = oc.price(spot=100, strike=105, expiry=0.5, rate=0.05, vol=true_vol, kind="call")

solved_vol = oc.iv(price=market_price, spot=100, strike=105, expiry=0.5, rate=0.05, kind="call")

print(f"True vol:   {true_vol:.10f}")
print(f"Solved vol: {solved_vol:.10f}")
print(f"Error:      {abs(solved_vol - true_vol):.2e}")

## 3. Greeks — All at Once

Price + Delta + Gamma + Theta + Vega + Rho in a single pass.

In [ ]:
g = oc.greeks(spot=100, strike=105, expiry=0.5, rate=0.05, vol=0.20, kind="call")

print(f"Price: ${g.price:.4f}")
print(f"Delta: {g.delta:.4f}    (position moves ${g.delta:.2f} per $1 spot move)")
print(f"Gamma: {g.gamma:.4f}    (delta changes {g.gamma:.4f} per $1 spot move)")
print(f"Theta: {g.theta:.4f}   (loses ${abs(g.theta):.4f} per day)")
print(f"Vega:  {g.vega:.4f}    (gains ${g.vega:.4f} per 1% vol increase)")
print(f"Rho:   {g.rho:.4f}    (gains ${g.rho:.4f} per 1% rate increase)")

## 4. Vectorized — Price an Entire Strike Ladder

In [ ]:
strikes = np.arange(80, 121, dtype=float)

# All 41 prices in one call
prices = oc.price(spot=100, strike=strikes, expiry=0.5, rate=0.05, vol=0.20, kind="call")

# Or get a full Greeks table as a DataFrame
df = oc.greeks_table(spot=100, strike=strikes, expiry=0.5, rate=0.05, vol=0.20, kind="call")
df.head(10)

## 5. Visualization

In [ ]:
# Greeks profile: how delta changes with spot price
fig = oc.plot.greek(
    "delta", spot_range=(70, 130),
    strike=100, expiry=0.5, rate=0.05, vol=0.20, kind="both"
)

In [ ]:
# Payoff diagram: long straddle
fig = oc.plot.payoff([
    oc.Leg("call", strike=100, qty=1, premium=5.50),
    oc.Leg("put",  strike=100, qty=1, premium=4.80),
])

In [ ]:
# Payoff diagram: iron condor
fig = oc.plot.payoff([
    oc.Leg("put",  strike=90,  qty=-1, premium=1.50),
    oc.Leg("put",  strike=95,  qty=1,  premium=3.00),
    oc.Leg("call", strike=105, qty=1,  premium=3.50),
    oc.Leg("call", strike=110, qty=-1, premium=1.80),
], spot_range=(80, 120))

## Next Steps

- **Notebook 02**: Explore all Greeks in depth
- **Notebook 03**: Connect to Interactive Brokers and fetch live chains
- **Notebook 04**: Analyze IV smiles from real market data
- **Notebook 05**: Build and visualize multi-leg strategies